# Movie Recommendation using TF-IDF + Cosine Similarity

In [1]:
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Dataset

In [2]:
movies = [
    {"title": "Interstellar",      "description": "Astronauts travel through a wormhole in space to find a new planet for humanity."},
    {"title": "Wall-E",            "description": "A robot living alone on Earth joins a space adventure to save humankind."},
    {"title": "The Martian",       "description": "An astronaut is stranded on Mars and must survive while NASA plans a rescue."},
    {"title": "Ratatouille",       "description": "A rat in Paris dreams of becoming a chef and secretly controls a kitchen worker."},
    {"title": "Julie & Julia",     "description": "A woman cooks all recipes from Julia Child's cookbook and documents her food journey."},
    {"title": "Super Size Me",     "description": "A filmmaker eats only McDonald's fast food for 30 days and studies its health effects."},
    {"title": "Avengers: Endgame","description": "Superheroes travel through time to collect infinity stones and defeat Thanos."},
    {"title": "Gravity",           "description": "Two astronauts struggle to survive after their shuttle is destroyed in outer space."},
    {"title": "Chef",              "description": "A head chef launches a food truck and rediscovers his passion for cooking."},
    {"title": "Ex Machina",        "description": "A programmer tests an AI robot and questions its consciousness and freedom."},
]

## 2. Preprocessing

Lowercase → remove punctuation → remove stopwords → stem

In [3]:
STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "by", "from", "is", "are", "was", "were", "be", "been",
    "have", "has", "had", "do", "does", "did", "will", "would", "could",
    "it", "its", "he", "she", "they", "we", "you", "i", "his", "her",
    "this", "that", "as", "not", "so", "all", "into", "while", "after",
}

def stem(word):
    for suffix in ["ing", "tion", "ness", "ment", "ed", "ly", "er", "es", "s"]:
        if word.endswith(suffix) and len(word) - len(suffix) > 2:
            return word[:-len(suffix)]
    return word

def preprocess(text):
    text   = text.lower()
    text   = text.translate(str.maketrans("", "", string.punctuation))
    tokens = [stem(t) for t in text.split() if t not in STOPWORDS and len(t) > 1]
    return " ".join(tokens)

# Test
print(preprocess("An astronaut is stranded on Mars and must survive."))

astronaut strand mar must survive


## 3. TF-IDF Vectorization

In [4]:
corpus = [preprocess(m["title"] + " " + m["description"]) for m in movies]

vectorizer   = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

print(f"Matrix shape: {tfidf_matrix.shape}  (documents x vocabulary)")

Matrix shape: (10, 89)  (documents x vocabulary)


## 4. Search Function

In [5]:
def search(query, top_n=3):
    query_vec   = vectorizer.transform([preprocess(query)])
    scores      = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = scores.argsort()[::-1][:top_n]

    print(f"Query: '{query}'\n")
    for rank, idx in enumerate(top_indices, 1):
        print(f"  #{rank} {movies[idx]['title']}")
        print(f"     Score: {scores[idx]:.4f}")
        print(f"     {movies[idx]['description']}")
        print()

## 5. Results

In [6]:
search("space adventure with robots")

Query: 'space adventure with robots'

  #1 Wall-E
     Score: 0.4953
     A robot living alone on Earth joins a space adventure to save humankind.

  #2 Ex Machina
     Score: 0.1622
     A programmer tests an AI robot and questions its consciousness and freedom.

  #3 Interstellar
     Score: 0.1254
     Astronauts travel through a wormhole in space to find a new planet for humanity.



In [7]:
search("healthy food and cooking")

Query: 'healthy food and cooking'

  #1 Chef
     Score: 0.3731
     A head chef launches a food truck and rediscovers his passion for cooking.

  #2 Julie & Julia
     Score: 0.3224
     A woman cooks all recipes from Julia Child's cookbook and documents her food journey.

  #3 Super Size Me
     Score: 0.1330
     A filmmaker eats only McDonald's fast food for 30 days and studies its health effects.



In [8]:
# Try your own query
search("artificial intelligence robot")

Query: 'artificial intelligence robot'

  #1 Ex Machina
     Score: 0.2878
     A programmer tests an AI robot and questions its consciousness and freedom.

  #2 Wall-E
     Score: 0.2791
     A robot living alone on Earth joins a space adventure to save humankind.

  #3 Chef
     Score: 0.0000
     A head chef launches a food truck and rediscovers his passion for cooking.

